# Entrega 04: baselines e métricas

Seção de `Maria Dionila` (`mariadionila`). Esta seção usa a base diária do RU para calcular um **baseline trivial** e um **baseline operacional**, medir o erro na validação temporal por tipo de refeição e no total, e converter o erro em R$ a partir de um preço de referência documentado.


In [1]:
import sys
import io

# Força saída do kernel em UTF-8: no Windows, o stdout do kernel às vezes usa o
# codepage do console, que corrompe acentos nas mensagens impressas abaixo.
if sys.stdout.encoding is None or sys.stdout.encoding.lower() != "utf-8":
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="strict")
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding="utf-8", errors="strict")

import numpy as np
import pandas as pd
from pathlib import Path


def achar_raiz():
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "dados" / "bruto" / "mock" / "catraca_ru.csv").is_file():
            return pasta
    raise FileNotFoundError(
        "Não encontrei dados/bruto/mock/catraca_ru.csv. Abra o notebook dentro do projeto."
    )


RAIZ = achar_raiz()
PASTA_MOCK = RAIZ / "dados" / "bruto" / "mock"
PASTA_PROCESSADO = RAIZ / "dados" / "processado"
REFEICOES = ["cafe_da_manha", "almoco", "jantar"]

print("Raiz do projeto localizada (contém dados/bruto/mock/catraca_ru.csv).")

Raiz do projeto localizada (contém dados/bruto/mock/catraca_ru.csv).


## 1. Base diária

Lê apenas `catraca_ru.csv`, mantém só os eventos `servida`, conta eventos distintos por `data` e `tipo_refeicao`, completa a grade de datas (preenchendo dias sem evento com zero, inclusive domingos, em que o RU simulado não abre) e salva a base em `dados/processado/base_analitica_diaria.csv`. A reconciliação confere que a soma da base agregada é igual ao total de eventos servidos.

In [2]:
catraca = pd.read_csv(PASTA_MOCK / "catraca_ru.csv", dtype={"id_matricula": "string"})
catraca["data_hora_refeicao"] = pd.to_datetime(catraca["data_hora_refeicao"], errors="coerce")
catraca["data"] = catraca["data_hora_refeicao"].dt.normalize()

if catraca["data"].isna().any():
    raise ValueError("Há datas inválidas em data_hora_refeicao.")
if catraca["id_registro_utilizacao"].duplicated().any():
    raise ValueError("id_registro_utilizacao tem duplicatas na catraca.")

servidas = catraca.loc[catraca["situacao_evento"].eq("servida")].copy()
print("Eventos servidos:", servidas["id_registro_utilizacao"].nunique())

base_diaria = (
    servidas.groupby(["data", "tipo_refeicao"])["id_registro_utilizacao"]
    .nunique()
    .reset_index(name="quantidade_servida")
    .sort_values(["data", "tipo_refeicao"])
)

total_esperado = servidas["id_registro_utilizacao"].nunique()
total_agregado = int(base_diaria["quantidade_servida"].sum())
if total_agregado != total_esperado:
    raise ValueError(
        f"Total agregado ({total_agregado:,}) diferente do total servido ({total_esperado:,})."
    )
print("Reconciliação ok: total agregado == total de eventos servidos.")

datas = pd.date_range(start=servidas["data"].min(), end=servidas["data"].max(), freq="D")
refeicoes = sorted(servidas["tipo_refeicao"].dropna().unique())
grade = pd.MultiIndex.from_product([datas, refeicoes], names=["data", "tipo_refeicao"])

base_diaria_completa = (
    base_diaria.set_index(["data", "tipo_refeicao"]).reindex(grade, fill_value=0).reset_index()
)
base_diaria_completa["quantidade_servida"] = base_diaria_completa["quantidade_servida"].astype("int64")
base_diaria_completa["dia_semana"] = base_diaria_completa["data"].dt.day_of_week
base_diaria_completa["mes"] = base_diaria_completa["data"].dt.month
base_diaria_completa["fim_de_semana"] = (base_diaria_completa["dia_semana"] >= 5).astype("int8")

print("\nDatas:", len(datas), "| Tipos de refeição:", len(refeicoes), "| Linhas:", len(base_diaria_completa))
print(base_diaria_completa.head(6).to_string(index=False))

PASTA_PROCESSADO.mkdir(parents=True, exist_ok=True)
caminho_saida = PASTA_PROCESSADO / "base_analitica_diaria.csv"
base_diaria_completa.to_csv(caminho_saida, index=False)

print("\nBase salva em:", caminho_saida.relative_to(RAIZ))

Eventos servidos: 221600
Reconciliação ok: total agregado == total de eventos servidos.

Datas: 2875 | Tipos de refeição: 3 | Linhas: 8625
      data tipo_refeicao  quantidade_servida  dia_semana  mes  fim_de_semana
2019-01-01        almoco                  15           1    1              0
2019-01-01 cafe_da_manha                  10           1    1              0
2019-01-01        jantar                   9           1    1              0
2019-01-02        almoco                  16           2    1              0
2019-01-02 cafe_da_manha                  11           2    1              0
2019-01-02        jantar                  10           2    1              0

Base salva em: dados\processado\base_analitica_diaria.csv


## 2. Partição temporal

Cortes por data, sem sorteio, mantendo a mesma fronteira usada na auditoria: treino até **31/12/2024**, validação até **31/12/2025**, teste a partir de **01/01/2026**. O conjunto de teste não é usado nesta seção; ele fica reservado para a avaliação final do projeto, depois que um modelo for ajustado.

In [3]:
FIM_TREINO = pd.Timestamp("2024-12-31")
FIM_VALIDACAO = pd.Timestamp("2025-12-31")

base = base_diaria_completa.copy()
base["conjunto"] = np.select(
    [base["data"] <= FIM_TREINO, base["data"] <= FIM_VALIDACAO],
    ["treino", "validacao"],
    default="teste",
)

resumo_particao = (
    base.groupby("conjunto")
    .agg(inicio=("data", "min"), fim=("data", "max"), dias=("data", "nunique"))
    .loc[["treino", "validacao", "teste"]]
)
print(resumo_particao)

treino = base.loc[base["conjunto"] == "treino"].copy()
validacao = base.loc[base["conjunto"] == "validacao"].copy()
teste = base.loc[base["conjunto"] == "teste"].copy()

if not (treino["data"].max() < validacao["data"].min() < teste["data"].min()):
    raise ValueError("A partição temporal ficou fora de ordem ou sobreposta.")
print("\nPartição em ordem: treino < validação < teste, sem sobreposição.")

              inicio        fim  dias
conjunto                             
treino    2019-01-01 2024-12-31  2192
validacao 2025-01-01 2025-12-31   365
teste     2026-01-01 2026-11-14   318

Partição em ordem: treino < validação < teste, sem sobreposição.


## 3. Baselines

**Cenário de uso imitado pela partição:** a cada véspera, a equipe do RU decide a quantidade de refeições a preparar no dia seguinte, por tipo de refeição, usando apenas o que já aconteceu até o fim do dia anterior; o treino representa o histórico disponível, a validação representa dias futuros ainda não vistos no ajuste, e o teste fica reservado para a comparação final.

### 3.1 Baseline trivial

Classe majoritária não se aplica a uma variável contínua de contagem; uso a **média do treino por tipo de refeição** como baseline trivial. Essa média ignora data, dia da semana e qualquer padrão sazonal; serve apenas como piso de comparação.

### 3.2 Baseline operacional

A regra operacional usada hoje por quem organiza a fila, sem qualquer modelo, é olhar para a **mesma data da semana anterior** (defasagem de 7 dias, `t-7`) e repetir a quantidade servida nesse dia como previsão para `t`. Essa informação está disponível no fim da véspera de `t-7` e, portanto, também no fim da véspera de `t`; não usa nenhuma informação do próprio dia previsto. A defasagem de 7 dias também captura o padrão semanal (por exemplo, domingos fechados, com demanda zero).

In [4]:
media_treino = treino.groupby("tipo_refeicao")["quantidade_servida"].mean()
print("Baseline trivial — média do treino por tipo de refeição:")
print(media_treino.round(2))

validacao = validacao.sort_values(["tipo_refeicao", "data"]).reset_index(drop=True)
validacao["previsto_trivial"] = validacao["tipo_refeicao"].map(media_treino)

serie_completa = base.sort_values(["tipo_refeicao", "data"]).reset_index(drop=True)
serie_completa["previsto_operacional"] = serie_completa.groupby("tipo_refeicao")["quantidade_servida"].shift(7)

previsao_operacional_validacao = serie_completa.loc[
    serie_completa["conjunto"] == "validacao",
    ["data", "tipo_refeicao", "previsto_operacional"],
]

comparacao = validacao.merge(previsao_operacional_validacao, on=["data", "tipo_refeicao"], how="left")

linhas_sem_lag = comparacao["previsto_operacional"].isna().sum()
print(f"\nLinhas da validação sem defasagem de 7 dias disponível: {linhas_sem_lag}")
# Como o treino termina em 31/12/2024 e a validação começa em 01/01/2025,
# a defasagem de 7 dias de todo dia da validação cai dentro do treino ou da própria validação já calculada,
# então nenhuma linha fica sem previsão operacional.

comparacao["erro_trivial"] = comparacao["quantidade_servida"] - comparacao["previsto_trivial"]
comparacao["erro_operacional"] = comparacao["quantidade_servida"] - comparacao["previsto_operacional"]

comparacao.head(10)[["data", "tipo_refeicao", "quantidade_servida", "previsto_trivial", "previsto_operacional"]]

Baseline trivial — média do treino por tipo de refeição:
tipo_refeicao
almoco           37.45
cafe_da_manha    19.87
jantar           18.00
Name: quantidade_servida, dtype: float64

Linhas da validação sem defasagem de 7 dias disponível: 0


,data,tipo_refeicao,quantidade_servida,previsto_trivial,previsto_operacional
0,2025-01-01,almoco,10,37.452099,13.0
1,2025-01-02,almoco,13,37.452099,14.0
2,2025-01-03,almoco,18,37.452099,13.0
3,2025-01-04,almoco,11,37.452099,13.0
4,2025-01-05,almoco,0,37.452099,0.0
5,2025-01-06,almoco,11,37.452099,13.0
6,2025-01-07,almoco,12,37.452099,20.0
7,2025-01-08,almoco,15,37.452099,10.0
8,2025-01-09,almoco,16,37.452099,13.0
9,2025-01-10,almoco,11,37.452099,18.0


## 4. Métrica na validação temporal

Métrica principal: **MAE** (erro absoluto médio, na mesma unidade da contagem de refeições), por ser direto de explicar à equipe e de converter em R$. Reporto também o **MAPE** (erro percentual) como leitura relativa e o **RMSE** como referência de dispersão. Dias com `quantidade_servida` igual a zero (domingos, em que o RU simulado não abre) entram no MAE e no RMSE normalmente, mas ficam fora da média do MAPE, porque o erro percentual não é definido quando o real é zero; esses dias são registrados à parte.

In [5]:
def calcular_metricas(dados, coluna_erro, coluna_real="quantidade_servida"):
    erro = dados[coluna_erro]
    erro_absoluto = erro.abs()
    mae = erro_absoluto.mean()
    rmse = np.sqrt((erro ** 2).mean())
    com_real_positivo = dados[coluna_real] > 0
    mape = (erro_absoluto[com_real_positivo] / dados.loc[com_real_positivo, coluna_real]).mean() * 100
    return pd.Series({
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MAPE_%": round(mape, 1),
        "dias_reais_zero": int((~com_real_positivo).sum()),
        "n_dias": len(dados),
    })


metricas_trivial_por_refeicao = comparacao.groupby("tipo_refeicao", group_keys=True).apply(
    lambda g: calcular_metricas(g, "erro_trivial"), include_groups=False
)
metricas_operacional_por_refeicao = comparacao.groupby("tipo_refeicao", group_keys=True).apply(
    lambda g: calcular_metricas(g, "erro_operacional"), include_groups=False
)

metricas_trivial_total = calcular_metricas(comparacao, "erro_trivial")
metricas_operacional_total = calcular_metricas(comparacao, "erro_operacional")

print("=== Baseline trivial (média do treino) — por tipo de refeição ===")
print(metricas_trivial_por_refeicao)
print("\n=== Baseline operacional (t-7) — por tipo de refeição ===")
print(metricas_operacional_por_refeicao)
print("\n=== Total (todas as refeições agrupadas) ===")
print(pd.DataFrame({"trivial": metricas_trivial_total, "operacional": metricas_operacional_total}))

=== Baseline trivial (média do treino) — por tipo de refeição ===
                 MAE   RMSE  MAPE_%  dias_reais_zero  n_dias
tipo_refeicao                                               
almoco         26.75  27.72    94.5             52.0   365.0
cafe_da_manha  13.19  14.20    70.9             52.0   365.0
jantar         10.94  12.10    57.4             52.0   365.0

=== Baseline operacional (t-7) — por tipo de refeição ===
                MAE   RMSE  MAPE_%  dias_reais_zero  n_dias
tipo_refeicao                                              
almoco         7.76  14.14    33.7             52.0   365.0
cafe_da_manha  5.30   8.06    35.2             52.0   365.0
jantar         5.07   7.45    32.7             52.0   365.0

=== Total (todas as refeições agrupadas) ===
                 trivial  operacional
MAE                16.96         6.04
RMSE               19.29        10.33
MAPE_%             74.20        33.90
dias_reais_zero   156.00       156.00
n_dias           1095.00      1095

## 5. Erro em R$

O mock não contém preços (ver `docs/contrato_previsao_mock.md` e `docs/dicionario_mock.md`). Para expressar o erro em R$, adoto um **preço de referência ilustrativo e documentado aqui**, não um preço oficial do RU/UFPB: **R$ 2,00 por refeição servida**, igual para café da manhã, almoço e jantar, inspirado no valor subsidiado usual de refeição em restaurantes universitários federais. Qualquer substituição por um preço real (ou preços diferentes por tipo de refeição) exige apenas trocar `PRECO_REFERENCIA` nesta célula; o restante do cálculo não muda. O valor em R$ é sempre `quantidade × preço de referência`, aplicado ao erro absoluto de cada baseline.

In [6]:
PRECO_REFERENCIA = 2.00  # R$ por refeição servida; pressuposto documentado acima, não é preço oficial.

comparacao["erro_reais_trivial"] = comparacao["erro_trivial"].abs() * PRECO_REFERENCIA
comparacao["erro_reais_operacional"] = comparacao["erro_operacional"].abs() * PRECO_REFERENCIA

erro_reais_por_refeicao = comparacao.groupby("tipo_refeicao").agg(
    MAE_reais_trivial=("erro_reais_trivial", "mean"),
    MAE_reais_operacional=("erro_reais_operacional", "mean"),
).round(2)

erro_reais_total = pd.Series({
    "MAE_reais_trivial": round(comparacao["erro_reais_trivial"].mean(), 2),
    "MAE_reais_operacional": round(comparacao["erro_reais_operacional"].mean(), 2),
})

print("Erro médio diário em R$, por tipo de refeição:")
print(erro_reais_por_refeicao)
print("\nErro médio diário em R$, total (todas as refeições agrupadas):")
print(erro_reais_total)

Erro médio diário em R$, por tipo de refeição:
               MAE_reais_trivial  MAE_reais_operacional
tipo_refeicao                                          
almoco                     53.50                  15.53
cafe_da_manha              26.37                  10.60
jantar                     21.88                  10.14

Erro médio diário em R$, total (todas as refeições agrupadas):
MAE_reais_trivial        33.92
MAE_reais_operacional    12.09
dtype: float64


## 6. Tabela de resultados

Resumo comparando os dois baselines na validação temporal (01/01/2025 a 31/12/2025), por tipo de refeição e no total.

In [7]:
tabela_resultados = pd.concat(
    {
        "trivial": metricas_trivial_por_refeicao.assign(
            MAE_reais=erro_reais_por_refeicao["MAE_reais_trivial"]
        ),
        "operacional": metricas_operacional_por_refeicao.assign(
            MAE_reais=erro_reais_por_refeicao["MAE_reais_operacional"]
        ),
    },
    names=["baseline"],
)

linha_total = pd.concat(
    {
        "trivial": metricas_trivial_total.to_frame().T.assign(
            MAE_reais=erro_reais_total["MAE_reais_trivial"]
        ).rename(index={0: "total"}),
        "operacional": metricas_operacional_total.to_frame().T.assign(
            MAE_reais=erro_reais_total["MAE_reais_operacional"]
        ).rename(index={0: "total"}),
    },
    names=["baseline"],
)
linha_total.index = linha_total.index.set_names(["baseline", "tipo_refeicao"])

tabela_resultados = pd.concat([tabela_resultados, linha_total]).sort_index(level=0)
caminho_tabela = PASTA_PROCESSADO / "resultados_baselines_validacao.csv"
tabela_resultados.to_csv(caminho_tabela)

print("Tabela salva em:", caminho_tabela.relative_to(RAIZ), "\n")
tabela_resultados

Tabela salva em: dados\processado\resultados_baselines_validacao.csv 



MAE   RMSE  MAPE_%  dias_reais_zero  n_dias  \
baseline    tipo_refeicao                                                  
operacional almoco          7.76  14.14    33.7             52.0   365.0   
            cafe_da_manha   5.30   8.06    35.2             52.0   365.0   
            jantar          5.07   7.45    32.7             52.0   365.0   
            total           6.04  10.33    33.9            156.0  1095.0   
trivial     almoco         26.75  27.72    94.5             52.0   365.0   
            cafe_da_manha  13.19  14.20    70.9             52.0   365.0   
            jantar         10.94  12.10    57.4             52.0   365.0   
            total          16.96  19.29    74.2            156.0  1095.0   

                           MAE_reais  
baseline    tipo_refeicao             
operacional almoco             15.53  
            cafe_da_manha      10.60  
            jantar             10.14  
            total              12.09  
trivial     almoco             53.50  
            cafe_da_manha      26.37  
            jantar             21.88  
            total              33.92

## 7. Explicação dos erros por tipo de refeição e no total

- **No total**, o baseline operacional (defasagem de 7 dias) reduz o MAE de cerca de 17 para cerca de 6 refeições por dia em relação ao baseline trivial, e o MAPE cai de aproximadamente 74% para 34%. A diferença mostra que a maior parte do erro do baseline trivial vem de padrões previsíveis (dia da semana, inclusive domingo fechado) que a média constante ignora e que a defasagem de 7 dias captura.
- **Almoço** tem o maior MAE absoluto nos dois baselines, porque também tem a maior quantidade média servida por dia; em termos relativos (MAPE), café da manhã e jantar não ficam muito atrás, porque têm quantidades menores e qualquer erro absoluto pesa proporcionalmente mais.
- O baseline trivial erra sistematicamente nos dois extremos: superestima a demanda nos domingos (quantidade real zero, baseline prevê a média) e subestima nos dias de maior movimento, porque a média do treino mistura dias letivos, recessos e domingos fechados sem diferenciá-los.
- O baseline operacional ainda carrega o efeito do crescimento da demanda ao longo do tempo, já registrado por Kaio Vitor (a média de servidas por dia no teste é cerca de 14% maior que no treino): como ele usa apenas o valor de 7 dias atrás, tende a ficar sistematicamente abaixo da demanda em períodos de crescimento consistente, mesmo capturando bem o padrão semanal.
- Em R$, ao preço de referência de R$ 2,00 por refeição, o erro diário médio do baseline operacional somado nas três refeições fica em torno de R$ 36 por dia, contra cerca de R$ 102 por dia do baseline trivial — a diferença dá a ordem de grandeza do que um modelo precisaria melhorar para justificar sua complexidade.

## 8. Pressupostos, limitações e comparação

**Pressupostos**

- A previsão é feita no fim do dia `D-1`, com tudo o que já aconteceu até então; a defasagem de 7 dias usa apenas valores já observados nessa janela, sem olhar para o próprio dia `D` nem para dias futuros.
- Dias sem evento (incluindo todos os domingos, em que o RU simulado não abre) entram na base diária e nos baselines como demanda igual a zero, e não como dado ausente. Essa é uma decisão herdada da etapa da base analítica: pressupõe que a ausência de evento significa demanda zero, não falha de registro.
- O preço de referência de R$ 2,00 por refeição (seção 5) é um pressuposto documentado para fins didáticos; não é um preço oficial do RU/UFPB, e o mock não contém preços.
- Os cortes da partição temporal (treino até 31/12/2024, validação até 31/12/2025) são os mesmos propostos por Kaio Vitor e ainda são provisórios, sujeitos a confirmação do grupo.

**Limitações**

- Esta seção reproduz, de forma resumida, etapas que são responsabilidade de outras integrantes (base analítica de Bianca, partição de Kaio Vitor) porque as branches correspondentes ainda não foram integradas a `develop/mock`. A reprodução usa a mesma lógica e os mesmos parâmetros documentados nas respectivas seções, mas não substitui a revisão e a integração definitivas feitas por elas.
- Os dois baselines são simples por definição; nenhum dos dois usa variáveis de calendário (fase letiva, recesso), vínculo ou auxílio, que podem reduzir ainda mais o erro em um modelo posterior.
- O conjunto de teste não foi tocado nesta seção, conforme a regra de só usá-lo na avaliação final.
- A base é inteiramente sintética; os valores de erro e de R$ não devem ser lidos como desempenho esperado com dados reais do RU/UFPB.

**Comparação**

O baseline operacional (defasagem de 7 dias) supera o baseline trivial (média do treino) em todas as métricas, por tipo de refeição e no total, porque incorpora o padrão semanal que o mock gera deliberadamente (variação por dia da semana e fechamento aos domingos). Qualquer modelo desenvolvido depois deve superar o baseline operacional, não apenas o trivial, para justificar sua complexidade.

## 9. Diagnóstico de vazamento (verificação individual)

Três variáveis relacionadas à construção dos baselines desta seção, classificadas em relação ao corte no fim do dia `D-1`:

| Variável | Classificação | Justificativa |
|---|---|---|
| `quantidade_servida` em `D-7` (defasagem de 7 dias, usada no baseline operacional) | Permitida | O valor de 7 dias atrás já aconteceu e foi registrado muito antes do fim de `D-1`; não usa nenhuma informação do dia previsto. |
| `quantidade_servida` do próprio dia `D` | Proibida por vazamento temporal | É o alvo da previsão; só existe depois que o dia `D` termina e as refeições já foram servidas, não no momento em que a previsão seria feita. |
| Quantidade de tentativas negadas (`situacao_evento = negada`) no dia `D` | Proibida por consequência da decisão | Negadas em `D` podem ocorrer porque a quantidade preparada seguiu o baseline e foi insuficiente. Usá-las para corrigir o próprio baseline faria ele aprender o efeito da sua decisão anterior, não a demanda real. |

## 10. Uso de IA generativa

Usei o Claude como apoio para organizar esta seção do notebook, redigir as explicações em markdown e revisar o cálculo das métricas.